# Alice & Bob resource estimator tutorial

This Jupyter notebook is a tutorial for the **Alice & Bob resource estimator**, a tool designed to estimate the physical resources required to execute quantum algorithms on Alice & Bob hardware.

The tutorial introduces the estimator’s concepts, workflow, and usage through concrete examples, with a focus on elliptic curve cryptography (ECC).

# Table of Contents
1. [Prerequisites](#Prerequisites)
2. [What Is the Alice & Bob Resource Estimator?](#What-Is-the-Alice-&-Bob-Resource-Estimator?)
3. [Installation](#Installation)
4. [Examples](#Examples)
   2. [ECC Based on Logical Counts](#ECC-based-on-logical-counts)
   3. [Estimation from a Qualtran bloq](#Estimation-from-a-Qualtran-bloq)
   4. [Estimation from a Q# File](#Estimation-from-a-Q#-file)


## Prerequisites

The Alice & Bob resource estimator provides a Python interface, and no knowledge of Rust is required. However, readers are expected to have the following background:

- Familiarity with basic Python syntax
- A basic understanding of quantum circuits and common quantum gates
- A basic understanding of cat qubits
- Some familiarity with quantum error correction (QEC)

While **prior knowledge of elliptic curve cryptography** is helpful for interpreting the example results, it is **not required** to use the resource estimator itself.


## What Is the Alice & Bob Resource Estimator?

The [Microsoft Azure Quantum Resource Estimator](https://arxiv.org/abs/2311.05801) introduces a layered framework that connects high-level quantum programs to physical resource estimates.

The **Alice & Bob resource estimator** builds on this framework by specializing the hardware and error-correction layers for Alice & Bob’s **cat-qubit architecture**.
It takes into account routing, quantum error correction, and magic-state factory implementations.


## Installation

See Readme.md

In [ ]:
#!pixi install
# You might also need to run the following command to build the rust dependencies
#!pixi run maturin develop --uv

Now you can import the resource estimator just like any other Python package.

In [ ]:
# Import the Rust functionality as a Python module
import anb_estimator

# Examples

## ECC based on logical counts

To obtain the logical resource counts, we reproduce the calculations described in [arXiv:2302.06639](https://arxiv.org/abs/2302.06639).
Physical estimate is then obtained from it.

In [ ]:
import math

bit_size = 256
window_size = 18

qubit_count = 9 * bit_size + window_size + 4

# Asymptotic gate counts, arXiv:2302.06639 (p. 21, app C.10)
cx_count = math.ceil(448 * bit_size**3 / window_size)
ccx_count = math.ceil(348 * bit_size**3 / window_size)

elliptic_curve = anb_estimator.estimate_logical_counts(  # type: ignore
    qubit_count,
    cx_count,
    ccx_count,
    frontier=False,
    error_budget=(0.333 * 0.5, 0.333 * 0.5, 0.0),
)

In [ ]:
print("=== Logical_estimates example ===")
print(elliptic_curve)

You can access one field individually:

In [ ]:
code_distance = elliptic_curve.code_distance
print(f"Code distance: {code_distance}")
# Same for other parameters

## Estimation from a Qualtran bloq

Another option is to use [Qualtran](https://qualtran.readthedocs.io). Qualtran then produces an estimate of the logical resource counts, which can subsequently be used as input to the resource estimator.

The resource estimation can then be run by using the function ``estimate_from_qualtran`` as shown below.

The cool thing about Qualtran is that with this interface we can easily perform a resource estimation for the large number of algorithms that are contained in the Qualtran library.
Here is another example for the QFT on a register of 100 qubits.

In [ ]:
from qualtran.bloqs.qft import QFTTextBook

qft_text_book = QFTTextBook(100)


result_qft = anb_estimator.estimate_from_qualtran(
    qft_text_book, frontier=False, error_budget=(0.333 * 0.5, 0.333 * 0.5, 0.0)
)

print(result_qft)

## Estimation from a Q# file

The last option is to specify an algorithm in a Q# file and pass it as an input to the resource estimator.
To perform the optimization we simply have to provide the path to the Q# file we are interested in. An example is given in ``qsharp/Adder.rs``.

In [ ]:
filename = "../qsharp/Adder.qs"
single_qsharp, frontier, counts = anb_estimator.estimate_qsharp_file(  # type: ignore
    filename, frontier=True, error_total=None, error_budget=(0.0005, 0.0005, 0.0)
)
print("=== Q# example ===")
print(single_qsharp)